In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn import metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


In [32]:
df =  pd.read_csv('titanic.csv')
df.head()

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   891 non-null    int64  
 1   PassengerId  891 non-null    int64  
 2   Survived     891 non-null    int64  
 3   Pclass       891 non-null    object 
 4   Name         891 non-null    object 
 5   Sex          891 non-null    object 
 6   Age          714 non-null    float64
 7   SibSp        891 non-null    int64  
 8   Parch        891 non-null    int64  
 9   Ticket       891 non-null    object 
 10  Fare         891 non-null    float64
 11  Cabin        204 non-null    object 
 12  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(6)
memory usage: 90.6+ KB


In [34]:
df.isna().sum() 

Unnamed: 0       0
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [35]:
df.describe()


,Unnamed: 0,PassengerId,Survived,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,445.000000,446.000000,0.383838,29.699118,0.523008,0.381594,32.204208
std,257.353842,257.353842,0.486592,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000
25%,222.500000,223.500000,0.000000,20.125000,0.000000,0.000000,7.910400
50%,445.000000,446.000000,0.000000,28.000000,0.000000,0.000000,14.454200
75%,667.500000,668.500000,1.000000,38.000000,1.000000,0.000000,31.000000
max,890.000000,891.000000,1.000000,80.000000,8.000000,6.000000,512.329200


In [36]:
# Lowercase all column names in df for uniformity
df.columns = df.columns.str.lower()

features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

X = df[features]
y = df[target]

print(f"Count distribution of the target column on the whole dataset: \n{y.value_counts()}")

Count distribution of the target column on the whole dataset: 
survived
0    549
1    342
Name: count, dtype: int64


#### Train-Test Split

In [37]:
from sklearn.model_selection import train_test_split

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)  # stratify preserve class distribution

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Training set size: 712
Testing set size: 179


In [38]:
# Alternative
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)  # stratify preserve class distribution

print(f"Count distribution of the target column on the train_set: \n{y_train.value_counts()}")
print(f"Count distribution of the target column on the test_set: \n{y_test.value_counts()}")

Count distribution of the target column on the train_set: 
survived
0    439
1    273
Name: count, dtype: int64
Count distribution of the target column on the test_set: 
survived
0    110
1     69
Name: count, dtype: int64


In [39]:
numeric_features = ["age", 'sibsp', 'parch', 'fare']
catergorical_features = ["pclass", "sex", "embarked"]

#### Numeric Pipeline

Steps:
1. Impute missing values using median
2. Apply StandardScaler

Why median?
- Robust to outliers

Why scaling?
KNN uses distance calculations, if features are not scaled , variables like fare could dominate Age

Scaling ensures all numeric features contribute equally

In [40]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Impute missing values with median
    ('scaler', StandardScaler())  # Standardize features
])

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Categorical preprocessing pipeline
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),   # fill missing values with most frequent category
    ("encoder", OneHotEncoder(handle_unknown="ignore"))     # convert categories to numeric, ignore unseen ones
])


#### ColumnTransformer

In [42]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, ["age", "fare"]),   # numeric columns
        ("cat", categorical_pipeline, ["sex", "embarked", "pclass"])  # categorical columns
    ]
)


In [43]:
# KNeighbors Pipeline
knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),  # Apply preprocessing steps
    ("classifier", KNeighborsClassifier(n_neighbors=1))  # KNN classifier
])

# Logistic Regression Pipeline
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),  # Apply preprocessing steps
    ("classifier", LogisticRegression(max_iter=1000))  # Logistic Regression classifier
])

# Random Forest Pipeline
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),  # Apply preprocessing steps
    ("classifier", RandomForestClassifier(random_state=42))  # Random Forest classifier
])

#### Comparing Multiple Models

We compare
- KNN
- Logistic Regression
- Random Forest

Use cross-validation on training data(to compare the models)

#### k-fold cross validation
- iteration 01
- iteration 02
- iteration 03
- iteration 04
- iteration 05

Then get the average 
- We compute mean accuracy fold
- The model with the highest average CV score is selected for tuning 
- This ensures we focus on the strongest candidate

In [49]:
from sklearn.model_selection import cross_val_score

# Compare models using 5-fold cross-validation
models = {
    "KNN": knn_pipeline,
    "Logistic Regression": logreg_pipeline,
    "Random Forest": rf_pipeline
}
results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring='accuracy'
    )
    # Correct indentation here
    results[name] = {
        "fold_scores": scores,
        "mean_score": scores.mean()
    }

    print(f"{name} Fold Accuracies: {scores}")
    print(f"{name} Mean CV Accuracy: {scores.mean():.4f}\n")

KNN Fold Accuracies: [0.76923077 0.73426573 0.71126761 0.78873239 0.77464789]
KNN Mean CV Accuracy: 0.7556

Logistic Regression Fold Accuracies: [0.78321678 0.75524476 0.8028169  0.83098592 0.80985915]
Logistic Regression Mean CV Accuracy: 0.7964

Random Forest Fold Accuracies: [0.79020979 0.72027972 0.80985915 0.80985915 0.83098592]
Random Forest Mean CV Accuracy: 0.7922



#### Selecting the Best Model
After computing cross-validation scores:
- We select the model with the highest mean performance.
- This becomes our candidate for hyperparameter tuning.
Why not tune all models?
- It is computationally expensive.
- Weak models rarely outperform strong ones even after tuning.


In [52]:
# Identify the best model based on mean accuracy
best_model_name = max(results, key=lambda m: results[m]["mean_score"])
print("\nBest Performing Model:", best_model_name)



Best Performing Model: Logistic Regression


In [50]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Best model from CV (Logistic Regression pipeline)
best_model = logreg_pipeline

# Train on the training set
best_model.fit(X_train, y_train)

# Predict on the test set
y_pred = best_model.predict(X_test)

# Evaluate performance
test_accuracy = accuracy_score(y_test, y_pred)
print(f'Logistic Regression Test Accuracy: {test_accuracy:.4f}\n')

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Logistic Regression Test Accuracy: 0.7709

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       110
           1       0.72      0.67      0.69        69

    accuracy                           0.77       179
   macro avg       0.76      0.75      0.75       179
weighted avg       0.77      0.77      0.77       179

Confusion Matrix:
[[92 18]
 [23 46]]


## Hyperparameter Tuning and Model Selection

In machine learning, there are two types of parameters:

- **Model parameters** → learned automatically during training  
- **Hyperparameters** → set manually before training  

Hyperparameters control how the learning process behaves.

### Examples
- **KNN** → `n_neighbors`, `metric`, `weights`  
- **Logistic Regression** → `C` (regularization strength)  
- **Random Forest** → `n_estimators`, `max_depth`  

### Why Hyperparameter Tuning Matters
Choosing the right hyperparameters can significantly improve model performance.

A model with default settings may:
- **Underfit** (too simple)  
- **Overfit** (too complex)  
- **Generalize poorly**  

In [53]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define parameter grid for Logistic Regression
param_grid = {
    'classifier__C': [0.1, 1.0, 10.0],              # Regularization strength
    'classifier__penalty': ['l2'],                  # L2 regularization
    'classifier__solver': ['lbfgs', 'liblinear']    # Different solvers
}

# Set up GridSearchCV
grid_search_logreg = GridSearchCV(
    logreg_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy"
)

# Fit on training data
grid_search_logreg.fit(X_train, y_train)

# Show best results
print("Best CV Score:", grid_search_logreg.best_score_)
print("Best Params:", grid_search_logreg.best_params_)

# Evaluate tuned model on test set
best_logreg = grid_search_logreg.best_estimator_
y_pred_tuned = best_logreg.predict(X_test)

test_accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Logistic Regression Test Accuracy: {test_accuracy_tuned:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Best CV Score: 0.7992317541613316
Best Params: {'classifier__C': 0.1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
Tuned Logistic Regression Test Accuracy: 0.7821

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.83       110
           1       0.74      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.77       179
weighted avg       0.78      0.78      0.78       179

Confusion Matrix:
[[94 16]
 [23 46]]


## RandomizedSearchCV

RandomizedSearch samples random combinations of hyperparameters.  
Instead of testing all possible combinations, it evaluates a fixed number (`n_iter`).  

### Advantages
- **Faster** than GridSearch  
- **Scales better** for large search spaces  

### Usage
Often applied in real-world scenarios with large models, where exhaustive search would be too computationally expensive.  


In [54]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define parameter distributions for Logistic Regression
param_distributions = {
    "classifier__C": np.logspace(-3, 3, 50),   # wide range of regularization strengths
    "classifier__penalty": ["l2"],             # L2 regularization
    "classifier__solver": ["lbfgs", "liblinear"]
}

# Set up RandomizedSearchCV
random_search_logreg = RandomizedSearchCV(
    logreg_pipeline,
    param_distributions,
    n_iter=20,              # number of random samples to try
    cv=5,
    scoring="accuracy",
    random_state=42
)

# Fit on training data
random_search_logreg.fit(X_train, y_train)

# Show best results
print("Best CV Score:", random_search_logreg.best_score_)
print("Best Params:", random_search_logreg.best_params_)

# Evaluate tuned model on test set
best_logreg = random_search_logreg.best_estimator_
y_pred_tuned = best_logreg.predict(X_test)

test_accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Logistic Regression Test Accuracy: {test_accuracy_tuned:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Best CV Score: 0.7992416034669556
Best Params: {'classifier__solver': 'liblinear', 'classifier__penalty': 'l2', 'classifier__C': 0.21209508879201905}
Tuned Logistic Regression Test Accuracy: 0.7821

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.83       110
           1       0.74      0.67      0.70        69

    accuracy                           0.78       179
   macro avg       0.77      0.76      0.77       179
weighted avg       0.78      0.78      0.78       179

Confusion Matrix:
[[94 16]
 [23 46]]
